## Practice Activity 7-1: Association Rules Mining

In the reading you analyzed transactions from a bakery. In this activity you will carry out a very similar analysis for purchases from a grocery store.

In [1]:
import pandas as pd

from plotnine import *

from mlxtend.frequent_patterns import apriori, association_rules

# to stop an annoying warning
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

## Groceries data set

- One month (30 days) of real point-of-sale transactions from a grocery store.
- 9,835 transactions (baskets).
- 169 product categories, such as whole milk, yogurt, bottled beer, etc.
- Source: the data set `Groceries` in the R package `arules`, provided by Hahsler, Hornik, and Reutterer. The copy we use is a plain-text version hosted on GitHub.


The file format is different from the bakery file. The bakery file had one row per item bought. This file has **one line per transaction**, and each line is a comma-separated list of the items in that basket. Different baskets have different numbers of items.

The next cell reads each line into a single text column. (Normal CSV reading expects every row to have the same number of columns, so we read each line as one string, using a separator `"|"` that never occurs in the file.)

In [2]:
import csv

raw = pd.read_csv(
    "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/groceries.csv",
    header=None,             # the file has no header row: line 1 is already a transaction
    names=["basket"],        # name the one column we create
    sep="|",                 # "|" never appears in this file, so each whole line lands in one column
    quoting=csv.QUOTE_NONE,  # treat any quote characters as ordinary text
)

In [3]:
raw

,basket
0,"citrus fruit,semi-finished bread,margarine,rea..."
1,"tropical fruit,yogurt,coffee"
2,whole milk
3,"pip fruit,yogurt,cream cheese ,meat spreads"
4,"other vegetables,whole milk,condensed milk,lon..."
...,...
9830,"sausage,chicken,beef,hamburger meat,citrus fru..."
9831,cooking chocolate
9832,"chicken,citrus fruit,other vegetables,butter,y..."
9833,"semi-finished bread,bottled water,soda,bottled..."


We'll start with the data in long format like in the reading.

- Make a column `basket_id` from the row number (`raw.index`).
- Split the text in `raw["basket"]` at the commas, using `.str.split(",")`. This produces a *list* of items in each row.
- Use `.explode(...)` on the column of lists to get one row per item.
- Keep only `basket_id` and `item`, and use `.reset_index(drop=True)`.
- Trim spaces

In [4]:
long = (
    raw
    .assign(basket_id=raw.index)                    # use the row number as the basket ID
    .assign(item=raw["basket"].str.split(","))      # "milk,bread" becomes ["milk", "bread"]
    .explode("item")                                # one row per (basket, item)
    [["basket_id", "item"]]                         # keep only the two columns we need
    .reset_index(drop=True)                         # renumber the rows 0, 1, 2, ...
)

long["item"] = long["item"].str.strip()             # remove stray leading/trailing spaces

long

,basket_id,item
0,0,citrus fruit
1,0,semi-finished bread
2,0,margarine
3,0,ready soups
4,1,tropical fruit
...,...,...
43362,9834,chicken
43363,9834,tropical fruit
43364,9834,other vegetables
43365,9834,vinegar


1. Build the transaction-by-item table `basket`. There should be one row per basket, one column per item, with `True` if the item is in the basket and `False` otherwise.

In [5]:
basket = pd.crosstab(long["basket_id"], long["item"]).astype(bool)
basket.shape, basket.head()

((9835, 169),
 item       Instant food products  UHT-milk  abrasive cleaner  \
 basket_id                                                      
 0                          False     False             False   
 1                          False     False             False   
 2                          False     False             False   
 3                          False     False             False   
 4                          False     False             False   
 
 item       artif. sweetener  baby cosmetics  baby food   bags  baking powder  \
 basket_id                                                                      
 0                     False           False      False  False          False   
 1                     False           False      False  False          False   
 2                     False           False      False  False          False   
 3                     False           False      False  False          False   
 4                     False           Fals

2.  Compute the size (number of items) of each basket and store it in `basket_sizes`.

In [6]:
basket_sizes = basket.sum(axis=1)
basket_sizes.head()

basket_id
0    4
1    3
2    1
3    4
4    4
dtype: int64

3. Summarize the distribution of the number of items.

In [7]:
basket_sizes.describe().round(2)

count    9835.00
mean        4.41
std         3.59
min         1.00
25%         2.00
50%         3.00
75%         6.00
max        32.00
dtype: float64

4. Compute the support of every item, sorted from largest to smallest, and store it in `item_support`. Then make a horizontal bar chart of the 15 most popular items.

In [8]:
item_support = basket.mean().sort_values(ascending=False)
support_plot = (
    item_support.head(15).sort_values().rename("support").reset_index()
)
support_plot.columns = ["item", "support"]
item_support.head(15), (
    ggplot(support_plot, aes(x="item", y="support"))
    + geom_col()
    + coord_flip()
    + labs(title="15 most popular grocery items", x="", y="support")
)

(item
 whole milk          0.255516
 other vegetables    0.193493
 rolls/buns          0.183935
 soda                0.174377
 yogurt              0.139502
 bottled water       0.110524
 root vegetables     0.108998
 tropical fruit      0.104931
 shopping bags       0.098526
 sausage             0.093950
 pastry              0.088968
 citrus fruit        0.082766
 bottled beer        0.080529
 newspapers          0.079817
 canned beer         0.077682
 dtype: float64,
 <plotnine.ggplot.ggplot object at 0x00000282A2BD4050>)

5. Consider the basket sizes and the item supports. Which item is most popular, and in what percentage of baskets does it appear? Which items are most common? What does this suggest about the "popular items co-occur just by chance" problem discussed in the reading?

The most popular item is whole milk. It appears in about one quarter of all baskets. Other vegetables, rolls/buns, soda, yogurt, and bottled water also occur often. Because these items have high individual support, they can co-occur in many baskets by chance. Lift helps separate common products from product pairs that occur together more often than their individual supports predict.

6. Consider the rule **{yogurt} → {whole milk}**. Compute and interpret the confidence of this rule.


In [9]:
s_yogurt = support(["yogurt"], basket) if "support" in globals() else basket["yogurt"].mean()
s_yogurt_milk = basket[["yogurt", "whole milk"]].all(axis=1).mean()
confidence_yogurt_milk = s_yogurt_milk / s_yogurt
print(f"confidence = {confidence_yogurt_milk:.3f}")
print(f"About {confidence_yogurt_milk:.1%} of baskets containing yogurt also contain whole milk.")

confidence = 0.402
About 40.2% of baskets containing yogurt also contain whole milk.


7. Compute and interpret the **lift** of {yogurt} → {whole milk}.

In [10]:
s_milk = basket["whole milk"].mean()
lift_yogurt_milk = confidence_yogurt_milk / s_milk
print(f"lift = {lift_yogurt_milk:.3f}")
print(f"Baskets with yogurt are {lift_yogurt_milk:.2f} times as likely to contain whole milk as a typical basket.")

lift = 1.572
Baskets with yogurt are 1.57 times as likely to contain whole milk as a typical basket.


8. Now compute and interpret the confidence and lift for the rule {whole milk} → {yogurt}.

In [11]:
s_whole_milk_yogurt = s_yogurt_milk
confidence_milk_yogurt = s_whole_milk_yogurt / s_milk
lift_milk_yogurt = confidence_milk_yogurt / s_yogurt
print(f"confidence = {confidence_milk_yogurt:.3f}")
print(f"lift = {lift_milk_yogurt:.3f}")
print(f"About {confidence_milk_yogurt:.1%} of baskets containing whole milk also contain yogurt.")
print(f"The lift is {lift_milk_yogurt:.2f}, which is the same pair association measured in the reverse direction.")

confidence = 0.219
lift = 1.572
About 21.9% of baskets containing whole milk also contain yogurt.
The lift is 1.57, which is the same pair association measured in the reverse direction.


9. Write two functions.

- `support(items, onehot)`: returns the share of baskets that contain **all** of the items in the list `items`.
- `rule_metrics(antecedent, consequent, onehot)`: returns a Pandas Series with the `support`, `confidence`, and `lift` of the rule `antecedent -> consequent`. Both `antecedent` and `consequent` are *lists* of items.

Test the functions on `["yogurt"]` → `["whole milk"]`. The values should match the ones you computed earlier.

In [12]:
def support(items, onehot):
    return onehot[list(items)].all(axis=1).mean()

def rule_metrics(antecedent, consequent, onehot):
    union = list(antecedent) + list(consequent)
    rule_support = support(union, onehot)
    antecedent_support = support(antecedent, onehot)
    consequent_support = support(consequent, onehot)
    return pd.Series({
        "support": rule_support,
        "confidence": rule_support / antecedent_support,
        "lift": rule_support / (antecedent_support * consequent_support),
    })

rule_metrics(["yogurt"], ["whole milk"], basket).round(3)

support       0.056
confidence    0.402
lift          1.572
dtype: float64

10. Whole milk is in about a quarter of all baskets. Consider the rules `{X} → {whole milk}` for the five antecedents `X` in the next cell. Before running anything, predict: for which of these will the lift be **above 1**? For which might the confidence look decent but the lift be **close to or below 1**? Just think about this before proceeding.

In [13]:
antecedents = pd.Series(["yogurt", "rolls/buns", "bottled water", "soda", "other vegetables"])

11. Compute the support, confidence, and lift of `{X} → {whole milk}` for each of the five antecedents above, and display the results sorted by lift (largest first). Compare to your predictions. Hint: Use `apply` with `lambda` on the Series of antecedents.


In [14]:
milk_rules = antecedents.apply(
    lambda item: rule_metrics([item], ["whole milk"], basket)
).apply(pd.Series)
milk_rules.index = antecedents
milk_rules.sort_values("lift", ascending=False).round(3)

,support,confidence,lift
yogurt,0.056,0.402,1.572
other vegetables,0.075,0.387,1.514
bottled water,0.034,0.311,1.217
rolls/buns,0.057,0.308,1.205
soda,0.040,0.230,0.899


12. Run `apriori` on `basket` with `min_support=0.01` and store the result in `frequent_itemsets`. Then add a column `n_items` giving the number of items in each itemset.

In [15]:
frequent_itemsets = apriori(basket, min_support=0.01, use_colnames=True)
frequent_itemsets["n_items"] = frequent_itemsets["itemsets"].apply(len)
frequent_itemsets.shape, frequent_itemsets.head()

((333, 3),
     support                    itemsets  n_items
 0  0.033452       frozenset({UHT-milk})        1
 1  0.017692  frozenset({baking powder})        1
 2  0.052466           frozenset({beef})        1
 3  0.033249        frozenset({berries})        1
 4  0.026029      frozenset({beverages})        1)

13. Generate the rules with `make_rules`, keeping rules with confidence of at least 0.20. Store them in `rules` and print how many there are.

In [16]:
def make_rules(frequent_itemsets, min_confidence):
    return association_rules(
        frequent_itemsets, metric="confidence", min_threshold=min_confidence
    )

rules = make_rules(frequent_itemsets, min_confidence=0.20)
print(f"{len(rules)} rules")

234 rules


14. Make the rules easier to read. Create a copy `rules_display` of `rules` and add three columns:

- `antecedent`: the antecedent items as text, e.g. `"tropical fruit, yogurt"`
- `consequent`: the consequent items as text
- `transactions`: the number of baskets that contain the whole rule (support times the number of baskets, rounded to an integer)

Then display the columns `antecedent`, `consequent`, `support`, `confidence`, `lift`, `transactions` for the first 8 rules.

In [17]:
rules_display = rules.copy()
rules_display["antecedent"] = rules_display["antecedents"].apply(lambda s: ", ".join(sorted(s)))
rules_display["consequent"] = rules_display["consequents"].apply(lambda s: ", ".join(sorted(s)))
rules_display["transactions"] = (rules_display["support"] * len(basket)).round().astype(int)
show_columns = ["antecedent", "consequent", "support", "confidence", "lift", "transactions"]
rules_display[show_columns].head(8).round({"support": 3, "confidence": 3, "lift": 3})

,antecedent,consequent,support,confidence,lift,transactions
0,beef,other vegetables,0.020,0.376,1.943,194
1,beef,rolls/buns,0.014,0.260,1.412,134
2,beef,root vegetables,0.017,0.331,3.040,171
3,beef,whole milk,0.021,0.405,1.585,209
4,beef,yogurt,0.012,0.223,1.598,115
5,berries,other vegetables,0.010,0.309,1.596,101
6,berries,whole milk,0.012,0.355,1.388,116
7,berries,yogurt,0.011,0.318,2.280,104


15. Show the 10 rules with the highest **confidence**. What do the highest-confidence rules have in common? Why is confidence alone a poor guide here?

In [18]:
top_confidence = rules_display.sort_values("confidence", ascending=False).head(10)
display(top_confidence[show_columns].round({"support": 3, "confidence": 3, "lift": 3}))
print("The highest-confidence rules often point to common products such as whole milk. A common consequent can produce high confidence even when lift is close to 1, so confidence alone can overstate the association.")

,antecedent,consequent,support,confidence,lift,transactions
156,"citrus fruit, root vegetables",other vegetables,0.010,0.586,3.030,102
184,"root vegetables, tropical fruit",other vegetables,0.012,0.585,3.021,121
164,"curd, yogurt",whole milk,0.010,0.582,2.279,99
154,"butter, other vegetables",whole milk,0.011,0.574,2.245,113
223,"root vegetables, tropical fruit",whole milk,0.012,0.570,2.231,118
226,"root vegetables, yogurt",whole milk,0.015,0.563,2.203,143
166,"domestic eggs, other vegetables",whole milk,0.012,0.553,2.162,121
233,"whipped/sour cream, yogurt",whole milk,0.011,0.525,2.053,107
214,"rolls/buns, root vegetables",whole milk,0.013,0.523,2.047,125
172,"other vegetables, pip fruit",whole milk,0.014,0.518,2.025,133


The highest-confidence rules often point to common products such as whole milk. A common consequent can produce high confidence even when lift is close to 1, so confidence alone can overstate the association.


16. Show the 10 rules with the highest **lift**. Compare this list with the highest-confidence list. What kinds of products appear now? Look at the `support` and `transactions` columns: how much evidence is each of these rules based on?

In [19]:
top_lift = rules_display.sort_values("lift", ascending=False).head(10)
display(top_lift[show_columns].round({"support": 3, "confidence": 3, "lift": 3}))
print("The highest-lift rules include less common product combinations. Most have support near 0.01, which is about 100 baskets, so the lift values are based on modest evidence.")

,antecedent,consequent,support,confidence,lift,transactions
157,"citrus fruit, other vegetables",root vegetables,0.010,0.359,3.295,102
208,"other vegetables, yogurt",whipped/sour cream,0.010,0.234,3.267,100
185,"other vegetables, tropical fruit",root vegetables,0.012,0.343,3.145,121
2,beef,root vegetables,0.017,0.331,3.040,171
156,"citrus fruit, root vegetables",other vegetables,0.010,0.586,3.030,102
184,"root vegetables, tropical fruit",other vegetables,0.012,0.585,3.021,121
190,root vegetables,"other vegetables, whole milk",0.023,0.213,2.842,228
188,"other vegetables, whole milk",root vegetables,0.023,0.310,2.842,228
155,butter,"other vegetables, whole milk",0.011,0.207,2.771,113
163,"curd, whole milk",yogurt,0.010,0.385,2.761,99


The highest-lift rules include less common product combinations. Most have support near 0.01, which is about 100 baskets, so the lift values are based on modest evidence.


17. Create `interesting_rules`: the rules with **confidence of at least 0.30 and lift of at least 1.5**, sorted by lift (largest first).


In [20]:
interesting_rules = (
    rules_display[
        (rules_display["confidence"] >= 0.30)
        & (rules_display["lift"] >= 1.5)
    ]
    .sort_values("lift", ascending=False)
)
interesting_rules[show_columns].round({"support": 3, "confidence": 3, "lift": 3})

,antecedent,consequent,support,confidence,lift,transactions
157,"citrus fruit, other vegetables",root vegetables,0.010,0.359,3.295,102
185,"other vegetables, tropical fruit",root vegetables,0.012,0.343,3.145,121
2,beef,root vegetables,0.017,0.331,3.040,171
156,"citrus fruit, root vegetables",other vegetables,0.010,0.586,3.030,102
184,"root vegetables, tropical fruit",other vegetables,0.012,0.585,3.021,121
...,...,...,...,...,...,...
89,onions,whole milk,0.012,0.390,1.527,119
72,hygiene articles,whole milk,0.013,0.389,1.522,126
17,brown bread,whole milk,0.025,0.389,1.521,248
106,other vegetables,whole milk,0.075,0.387,1.514,736


18. Choose **one** rule from the table and interpret its **support**, **confidence**, and **lift** with numbers, as you did for {yogurt} → {whole milk}.

For the rule `citrus fruit, other vegetables -> root vegetables`, support is 0.010, so the full rule appears in about 102 of 9,835 baskets. Confidence is 0.359, so 35.9% of baskets containing citrus fruit and other vegetables also contain root vegetables. Lift is 3.295, so this combination occurs about 3.3 times as often as the individual supports would predict.

19. The manager wants to **increase sales of yogurt**. Which products should the store link to yogurt (shelf placement, a joint promotion, a recommendation)?

In [21]:
yogurt_links = (
    rules_display[rules_display["antecedent"] == "yogurt"]
    .sort_values(["lift", "transactions"], ascending=[False, False])
)
display(yogurt_links[show_columns].round({"support": 3, "confidence": 3, "lift": 3}))
print("Use the highest-lift products as promotion candidates, with transaction counts as a check on sample size.")

,antecedent,consequent,support,confidence,lift,transactions
144,yogurt,tropical fruit,0.029,0.210,2.000,288
107,yogurt,other vegetables,0.043,0.311,1.608,427
150,yogurt,whole milk,0.056,0.402,1.572,551
127,yogurt,rolls/buns,0.034,0.246,1.339,338


Use the highest-lift products as promotion candidates, with transaction counts as a check on sample size.


---
## References and attribution


- Hahsler, M., Gr&uuml;n, B., and Hornik, K. (2005). arules: A computational environment for mining association rules and frequent item sets. *Journal of Statistical Software*, 14(15). (The R package that distributes the data.)
- Data copy: GitHub repository `stedy/Machine-Learning-with-R-datasets` (`groceries.csv`), the data used in Lantz, B., *Machine Learning with R*.
- Raschka, S. (2018). MLxtend: Providing machine learning and data science utilities and extensions to Python's scientific computing stack. *Journal of Open Source Software*, 3(24), 638. <https://doi.org/10.21105/joss.00638>


**Preparation of this notebook:** drafted with the assistance of generative AI tools (ChatGPT and Claude).